# Prétraitement et rééchantillonnage

Ce notebook explore trois stratégies classiques pour compenser le
déséquilibre des classes observé dans `01_exploration_donnees.ipynb` :
sur-échantillonnage aléatoire, sous-échantillonnage aléatoire, et une méthode
combinée (SMOTEENN).

**Note.** Ces stratégies sont explorées ici à titre exploratoire. Le modèle
final (`03_regression_logistique.ipynb`) gère le déséquilibre directement via
le paramètre `class_weight='balanced'` de `LogisticRegression`, plutôt que par
rééchantillonnage : ce choix évite d'entraîner le modèle sur des échantillons
synthétiques ou dupliqués, plus coûteux et pas toujours bénéfiques sur ce jeu
de données. Ce notebook documente néanmoins les alternatives envisagées.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

In [2]:
CLASSES_NAMES = [
    "Avoine d'hiver", "Avoine de printemps", "Blé tendre d'hiver",
    "Blé tendre de printemps", "Colza d'hiver", "Luzerne déshydratée",
    "Luzerne", "Mélange de légumineuses et graminées", "Maïs",
    "Maïs ensilage", "Orge d'hiver", "Orge de printemps",
    "Prairie permanente - herbe",
    "Prairie permanente - ressources fourragères ligneuses",
    "Prairie en rotation longue", "Pomme de terre de consommation",
    "Prairie temporaire", "Soja", "Tournesol", "Triticale d'hiver",
]

DATA_DIR = Path("data")
Y_train = np.load(DATA_DIR / "train_labels.npy")

images_path = DATA_DIR / "train_images.npy"
if images_path.exists():
    X_train = np.load(images_path).astype(np.float32)
else:
    X_train = None
    print("Images non présentes localement : les cellules de rééchantillonnage "
          "ci-dessous nécessitent train_images.npy (voir data/NOTE_DONNEES.md).")

Images non présentes localement : les cellules de rééchantillonnage ci-dessous nécessitent train_images.npy (voir data/NOTE_DONNEES.md).


## Fonctions de rééchantillonnage

In [3]:
def class_counts_df(Y, classes_names=CLASSES_NAMES):
    counts = np.bincount(Y, minlength=len(classes_names))
    return pd.DataFrame({"classe": classes_names, "n_echantillons": counts})


def plot_class_counts(df, title):
    plt.figure(figsize=(12, 4))
    plt.bar(df["classe"], df["n_echantillons"])
    plt.xticks(rotation=60, ha="right")
    plt.ylabel("Nombre d'échantillons")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def apply_random_oversampling(X, Y):
    """Sur-échantillonne aléatoirement les classes minoritaires."""
    n, t, c, h, w = X.shape
    X_flat = X.reshape(n, -1)
    X_res, Y_res = RandomOverSampler(random_state=42).fit_resample(X_flat, Y)
    return X_res.reshape(-1, t, c, h, w), Y_res


def apply_random_undersampling(X, Y):
    """Sous-échantillonne aléatoirement les classes majoritaires."""
    n, t, c, h, w = X.shape
    X_flat = X.reshape(n, -1)
    X_res, Y_res = RandomUnderSampler(random_state=42).fit_resample(X_flat, Y)
    return X_res.reshape(-1, t, c, h, w), Y_res


def apply_combined_resampling(X, Y):
    """Combine sur- et sous-échantillonnage via SMOTEENN."""
    n, t, c, h, w = X.shape
    X_flat = X.reshape(n, -1)
    X_res, Y_res = SMOTEENN(random_state=42).fit_resample(X_flat, Y)
    return X_res.reshape(-1, t, c, h, w), Y_res

## Sur-échantillonnage aléatoire (Random Oversampling)

In [4]:
if X_train is not None:
    X_ros, Y_ros = apply_random_oversampling(X_train, Y_train)
    plot_class_counts(class_counts_df(Y_ros), "Après sur-échantillonnage aléatoire")

## Sous-échantillonnage aléatoire (Random Undersampling)

In [5]:
if X_train is not None:
    X_rus, Y_rus = apply_random_undersampling(X_train, Y_train)
    plot_class_counts(class_counts_df(Y_rus), "Après sous-échantillonnage aléatoire")

## Rééchantillonnage combiné (SMOTEENN)

In [6]:
if X_train is not None:
    X_comb, Y_comb = apply_combined_resampling(X_train, Y_train)
    plot_class_counts(class_counts_df(Y_comb), "Après SMOTEENN")